# FRED Silver Transformation
Reads raw observations from Bronze, fixes data types, adds human-readable series names, and flags data quality issues.

### Step 1: Read Bronze, clean types, add series names, flag quality

In [0]:
import pandas as pd

# Read Bronze Delta table
df_silver = spark.table("bronze_fred_economic").toPandas()

# Fix data types
df_silver['date'] = pd.to_datetime(df_silver['date'])
df_silver['value'] = pd.to_numeric(df_silver['value'], errors='coerce')

# Add human-readable series names
series_names = {
    "UNRATE": "Unemployment Rate",
    "CPIAUCSL": "Consumer Price Index",
    "FEDFUNDS": "Federal Funds Rate",
    "MORTGAGE30US": "30-Year Mortgage Rate",
    "GDP": "Gross Domestic Product"
}
df_silver['series_name'] = df_silver['series_id'].map(series_names)

# Data quality flag
df_silver['data_quality_flag'] = df_silver['value'].isna()

print(f"Silver records: {len(df_silver)}")
print(f"Records with missing values: {df_silver['data_quality_flag'].sum()}")
display(df_silver.head(10))

Silver records: 5960
Records with missing values: 6


series_id,date,value,series_name,data_quality_flag
MORTGAGE30US,2013-08-01T00:00:00.000Z,4.39,30-Year Mortgage Rate,false
MORTGAGE30US,2013-08-08T00:00:00.000Z,4.4,30-Year Mortgage Rate,false
MORTGAGE30US,2013-08-15T00:00:00.000Z,4.4,30-Year Mortgage Rate,false
MORTGAGE30US,2013-08-22T00:00:00.000Z,4.58,30-Year Mortgage Rate,false
MORTGAGE30US,2013-08-29T00:00:00.000Z,4.51,30-Year Mortgage Rate,false
MORTGAGE30US,2013-09-05T00:00:00.000Z,4.57,30-Year Mortgage Rate,false
MORTGAGE30US,2013-09-12T00:00:00.000Z,4.57,30-Year Mortgage Rate,false
MORTGAGE30US,2013-09-19T00:00:00.000Z,4.5,30-Year Mortgage Rate,false
MORTGAGE30US,2013-09-26T00:00:00.000Z,4.32,30-Year Mortgage Rate,false
MORTGAGE30US,2013-10-03T00:00:00.000Z,4.22,30-Year Mortgage Rate,false


### Step 2: Write cleaned data to Silver Delta table

In [0]:
df_silver_spark = spark.createDataFrame(df_silver)

df_silver_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_fred_economic")

print(f"Silver table written: {df_silver_spark.count()} records")

Silver table written: 5960 records
